# What to Freeze: PT-SAM Training (Part 1 of 2)

This is **Part 1** of a split Kaggle workflow for the SAM freeze-strategy benchmark.

**What this notebook does:** Trains only PT-SAM, the most time-intensive strategy (~5.2 hours on a T4 GPU). PT-SAM freezes the entire SAM model and learns 8 prompt tokens (~2,048 trainable parameters) over 312 epochs with heavy augmentation.

**Part 2** (`kaggle_benchmark_eval.ipynb`) will:
- Copy PT-SAM's trained checkpoint from this notebook's output
- Train MedSAM and PP-SAM
- Evaluate all strategies (including Base SAM zero-shot)
- Generate comparison tables and visualizations

**After this notebook completes**, save a version (File > Save Version or the Save button). Part 2 uses this notebook's saved output as a Kaggle input dataset.

**Repository:** [github.com/nethum529/what_to_freeze](https://github.com/nethum529/what_to_freeze)

## Environment Setup

In [ ]:
%pip install -q "monai>=1.3.0" "albumentations>=1.3.0"

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/what_to_freeze"

if not os.path.exists(REPO_DIR):
    # GIT_LFS_SKIP_SMUDGE=1 replaces all *.pth files with LFS pointers.
    # This is safe because: SAM checkpoint is downloaded separately (not from LFS),
    # and trained checkpoints (model_best.pth) are regenerated during training.
    env = os.environ.copy()
    env["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(
        ["git", "clone", "https://github.com/nethum529/what_to_freeze.git", REPO_DIR],
        env=env, check=True,
    )

# Install MedSAM (provides segment_anything package)
# --no-deps avoids CustomInstall triggering redundant monai/CUDA pip packages
%pip install -q --no-deps -e /kaggle/working/what_to_freeze/MedSAM-main

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import os
import subprocess

SAM_CKPT = "work_dir/SAM/sam_vit_b_01ec64.pth"
os.makedirs("work_dir/SAM", exist_ok=True)

if not os.path.exists(SAM_CKPT) and not os.path.islink(SAM_CKPT):
    # Option 1: Kaggle dataset mount (add "sam-vit-b" dataset to notebook)
    kaggle_paths = [
        "/kaggle/input/sam-vit-b/sam_vit_b_01ec64.pth",
        "/kaggle/input/sam-vit-b-01ec64/sam_vit_b_01ec64.pth",
        "/kaggle/input/sam-checkpoint/sam_vit_b_01ec64.pth",
    ]
    linked = False
    for p in kaggle_paths:
        if os.path.exists(p):
            os.symlink(p, SAM_CKPT)
            print(f"Linked SAM checkpoint from {p}")
            linked = True
            break

    # Option 2: Download from Meta AI (requires internet enabled in notebook settings)
    if not linked:
        print("Downloading SAM ViT-B checkpoint (~375 MB)...")
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", SAM_CKPT,
             "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"],
            check=True,
        )

assert os.path.exists(SAM_CKPT), f"SAM checkpoint not found at {SAM_CKPT}"
print(f"SAM checkpoint ready: {os.path.getsize(SAM_CKPT) / 1e6:.0f} MB")

In [ ]:
import os
import torch

SAM_CKPT = "work_dir/SAM/sam_vit_b_01ec64.pth"

# Check ETIS dataset
assert os.path.exists("data/ETIS-LaribPolypDB/images"), "ETIS images not found -- check git clone"
assert os.path.exists("data/ETIS-LaribPolypDB/masks"), "ETIS masks not found -- check git clone"
n_images = len([f for f in os.listdir("data/ETIS-LaribPolypDB/images") if f.endswith(".png")])
assert n_images == 196, f"Expected 196 ETIS images, got {n_images} -- clone may be incomplete"
print(f"ETIS dataset: {n_images} images")

# Check GPU
print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")
    print("Enable GPU: Settings > Accelerator > GPU T4")

# Check SAM checkpoint size
ckpt_mb = os.path.getsize(SAM_CKPT) / 1e6
print(f"\nSAM checkpoint: {ckpt_mb:.0f} MB")
assert ckpt_mb > 300, f"SAM checkpoint seems too small ({ckpt_mb:.0f} MB), may be corrupted"
print("\nAll prerequisites verified.")

## Data Preprocessing

Convert 196 raw ETIS PNG images and masks to 1024x1024 numpy arrays, then create a reproducible 160/36 train/test split (seed=42).

In [ ]:
import subprocess
subprocess.run(["python", "scripts/preprocess_etis.py"], check=True)

## PT-SAM Training

PT-SAM freezes the **entire** SAM model (image encoder, prompt encoder, mask decoder) and learns only **8 prompt tokens** (~2,048 trainable parameters). This is the most parameter-efficient strategy in the benchmark.

Training details:
- **312 epochs**, batch size 4, lr=0.05 with cosine annealing
- **Heavy augmentation** via albumentations (spatial + photometric transforms), so the encoder must run every batch (no embedding caching)
- **Loss:** weighted CE:Dice = 0.2:0.8
- **No early stopping** -- fixed epoch count for reproducibility
- **Estimated time:** ~5.2 hours on T4 GPU

The Kaggle session continues running in the background if you close the browser tab.

In [ ]:
import subprocess
subprocess.run(
    "set -o pipefail; python scripts/run_benchmark.py --strategy ptsam --skip-eval 2>&1 | tee ptsam_train.log",
    shell=True, check=True, executable="/bin/bash",
)

## Verify Training Output

After training completes, the following files should exist in `work_dir/benchmark_etis/ptsam/`:
- `model_best.pth` -- checkpoint with lowest validation loss
- `model_latest.pth` -- checkpoint from the final epoch
- `train_losses.npy` / `val_losses.npy` -- per-epoch loss arrays
- `train_result.json` -- training summary (params, timing, best loss)

In [ ]:
import os
import json

work_dir = "work_dir/benchmark_etis/ptsam"
required = ["model_best.pth", "model_latest.pth", "train_losses.npy", "val_losses.npy", "train_result.json"]

print("PT-SAM training outputs:")
for f in required:
    path = os.path.join(work_dir, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    status = f"OK ({size:.1f} MB)" if exists else "MISSING"
    print(f"  {f}: {status}")

# Show training summary
result_path = os.path.join(work_dir, "train_result.json")
if os.path.exists(result_path):
    with open(result_path) as f:
        result = json.load(f)
    print(f"\nTraining summary:")
    print(f"  Trainable params: {result.get('trainable_params', 'N/A')}")
    best_loss = result.get('best_val_loss')
    print(f"  Best val loss: {best_loss:.4f}" if best_loss is not None else "  Best val loss: N/A")
    print(f"  Total time: {result.get('total_time_seconds', 0)/60:.1f} min")

## Next Steps

1. **Save this notebook version** -- File > Save Version (or the Save button). This preserves the trained PT-SAM checkpoint in the notebook's output.
2. **Open Part 2** -- `kaggle_benchmark_eval.ipynb`
3. **Add this notebook's output as input** -- In Part 2, click "Add Data" and search for this notebook's saved output. Add it as a Kaggle input dataset.
4. **Part 2 will:**
   - Copy PT-SAM's trained checkpoint from the input dataset
   - Train MedSAM (~36 min) and PP-SAM (~2 hours)
   - Run Base SAM zero-shot evaluation (~2 min)
   - Evaluate all four strategies on the 36-image test set
   - Generate comparison tables, training curves, and metric visualizations